# MIVA-KNIGHT: Five Multimodal Machine Learning Challenges (CREMA-D / Pipeline D)
## Representation · Translation · Fusion · Co-learning · Alignment
### **Powered by `crema_cache.pt` + Pipeline D weights (`AudioProjection` / `VideoFrameProjection`)**

This notebook is a **CREMA-D variant** of `MIVA_KNIGHT_Challenges_Implementation.ipynb`, aligned with:
- `MIVA_KNIGHT_PipelineD_CREMAD_Annotated_fixed.ipynb` (InfoNCE + Stage-2 SupCon, caching, projection heads)
- `MIVA_KNIGHT_PipelineD_Crema_D_Phase_2_Annotated_fixed.ipynb` (Phase 3 SupCon refinement, centroid evaluation)

**Author:** Oluwakayode (Kayode) Soyinka | IT 581 Capstone | Concordia University of Edmonton  
**Supervisor:** Dr. Baidya Saha

---

### What changes vs the CMU-MOSI notebook
- **Data:** CREMA-D cached features (`Data/crema_embeddings_cache/crema_cache.pt`) — 768-d Wav2Vec audio + middle video frame tensor per clip; **6 emotion labels** (no sentiment regression).
- **Cross-modal focus:** Challenge 1 uses **audio ↔ video** retrieval (Pipeline D design) instead of text ↔ audio.
- **Fusion (C3):** A lightweight **audio+video** head trained on top of **frozen** projection weights (same spirit as multimodal fusion, adapted to CREMA-D).

### Expected Drive layout (matches Pipeline D notebooks)
```
Data/crema_embeddings_cache/crema_cache.pt
models/miva_knight_pipelineD/audio_projection_cremad.pth
models/miva_knight_pipelineD/video_frame_projection.pth   (required)
models/miva_knight_pipelineD/emotion_centroids_cremad.pt  (optional)
```

### Real data only
This notebook **requires** `crema_cache.pt` and both projection checkpoints on Drive. It does **not** synthesize cache entries or run random-init projections.



## Cell 0 — Setup: Drive, imports, paths, constants

Mirrors Pipeline D path discovery (`POSSIBLE_BASES`, shortcut targets) and `crema_cache.pt` location.


In [ ]:
# ── Standard imports ──────────────────────────────────────────
import json, math, os, random, re, warnings
from collections import Counter, defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── PyTorch 2.6 patch: apply once; keep real loader on torch (safe if cells re-run)
if getattr(torch, "_miva_original_load", None) is None:
    torch._miva_original_load = torch.load

def _patched_torch_load(f, *a, **kw):
    kw.setdefault("weights_only", False)
    return torch._miva_original_load(f, *a, **kw)

torch.load = _patched_torch_load
print("PyTorch 2.6 patch applied (weights_only=False default)")

# ── Google Drive + project base (from Pipeline D notebooks) ───
IN_COLAB = False
PROJECT_BASE = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    print("Not in Colab / Drive not mounted — set PROJECT_BASE manually if needed.")

POSSIBLE_BASES = [
    "/content/drive/MyDrive/ Oluwakayode Soyinka IT 581 Project ",
    "/content/drive/MyDrive/Oluwakayode Soyinka IT 581 Project",
]
shortcut_root = "/content/drive/.shortcut-targets-by-id"
if os.path.exists(shortcut_root):
    for sid in os.listdir(shortcut_root):
        sp = os.path.join(shortcut_root, sid)
        if not os.path.isdir(sp):
            continue
        for folder in os.listdir(sp):
            if "soyinka" in folder.lower() or "581" in folder.lower():
                POSSIBLE_BASES.append(os.path.join(sp, folder))

for path in POSSIBLE_BASES:
    if path and os.path.exists(path) and os.path.exists(os.path.join(path, "models")):
        PROJECT_BASE = path
        break

# Optional: set MIVA_PROJECT_BASE to your project root if Drive path not auto-detected
if PROJECT_BASE is None:
    _env = os.environ.get("MIVA_PROJECT_BASE", "").strip()
    PROJECT_BASE = _env if _env else None

EMBED_DIM = 512
WAV2VEC_DIM = 768
RESNET_DIM = 2048
BATCH_SIZE = 32
TAU = 0.07

if PROJECT_BASE:
    CACHE_FILE = os.path.join(PROJECT_BASE, "Data", "crema_embeddings_cache", "crema_cache.pt")
    PIPELINE_D_DIR = os.path.join(PROJECT_BASE, "models", "miva_knight_pipelineD")
    AUDIO_PROJ_PATH = os.path.join(PIPELINE_D_DIR, "audio_projection_cremad.pth")
    VIDEO_PROJ_PATH = os.path.join(PIPELINE_D_DIR, "video_frame_projection.pth")
    CENTROIDS_PATH = os.path.join(PIPELINE_D_DIR, "emotion_centroids_cremad.pt")
else:
    CACHE_FILE = PIPELINE_D_DIR = AUDIO_PROJ_PATH = VIDEO_PROJ_PATH = CENTROIDS_PATH = None

print(f"Device: {DEVICE}")
print(f"PROJECT_BASE: {PROJECT_BASE or '(not set — mount Drive)'}")
if CACHE_FILE:
    print(f"CACHE_FILE: {CACHE_FILE}  exists={os.path.exists(CACHE_FILE)}")

def _require_real_crema_assets():
    """Fail fast unless real cache and Pipeline D projection weights exist on disk."""
    if not PROJECT_BASE or not os.path.isdir(PROJECT_BASE):
        raise RuntimeError(
            'PROJECT_BASE not found. Mount Google Drive and ensure the project folder '
            'contains Data/ and models/, or set MIVA_PROJECT_BASE to that path.'
        )
    if not CACHE_FILE or not os.path.isfile(CACHE_FILE):
        raise RuntimeError(
            f'Real CREMA cache missing: {CACHE_FILE}. Run Pipeline D cache extraction first.'
        )
    if not AUDIO_PROJ_PATH or not os.path.isfile(AUDIO_PROJ_PATH):
        raise RuntimeError(
            f'Real weights missing: {AUDIO_PROJ_PATH}. Train/save Pipeline D audio_projection_cremad.pth.'
        )
    if not VIDEO_PROJ_PATH or not os.path.isfile(VIDEO_PROJ_PATH):
        raise RuntimeError(
            f'Real weights missing: {VIDEO_PROJ_PATH}. Train/save Pipeline D video_frame_projection.pth.'
        )
    ck = torch.load(CACHE_FILE, map_location='cpu')
    if len(ck) < 32:
        raise RuntimeError(f'crema_cache.pt has only {len(ck)} entries; expected at least 32 real clips.')
    sample = next(iter(ck.values()))
    if sample['audio_emb'].shape[-1] != 768:
        raise RuntimeError('Cache audio_emb must be 768-d (Wav2Vec).')
    return ck


crema_cache = _require_real_crema_assets()
print(f'Loaded real crema_cache: {len(crema_cache):,} clips')




## Cell 1 — Shared blocks: `AudioProjection` & `VideoFrameProjection`

Architectures match `MIVA_KNIGHT_PipelineD_CREMAD_Annotated_fixed.ipynb` (768→512 audio, ResNet-50 frame → 512 video).


In [ ]:
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights


class AudioProjection(nn.Module):
    def __init__(self, wav2vec_dim=768, embed_dim=512):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(wav2vec_dim, wav2vec_dim),
            nn.GELU(),
            nn.LayerNorm(wav2vec_dim),
            nn.Linear(wav2vec_dim, embed_dim),
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        return F.normalize(self.norm(self.proj(x)), p=2, dim=1)


class VideoFrameProjection(nn.Module):
    def __init__(self, embed_dim=512, freeze_backbone=True):
        super().__init__()
        resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.proj = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.LayerNorm(1024),
            nn.Linear(1024, embed_dim),
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, images):
        with torch.no_grad():
            feat = self.backbone(images).flatten(1)
        return F.normalize(self.norm(self.proj(feat)), p=2, dim=1)


CREMA_EMOTIONS_6 = ["angry", "disgust", "fearful", "happy", "neutral", "sad"]
CREMA_EMOTION_TO_LABEL = {e: i for i, e in enumerate(CREMA_EMOTIONS_6)}
CREMA_LABEL_TO_EMOTION = {i: e for i, e in enumerate(CREMA_EMOTIONS_6)}

audio_proj = AudioProjection(WAV2VEC_DIM, EMBED_DIM).to(DEVICE)
video_proj = VideoFrameProjection(EMBED_DIM, freeze_backbone=True).to(DEVICE)

ck = torch.load(AUDIO_PROJ_PATH, map_location=DEVICE)
sd = ck.get("model_state_dict", ck)
audio_proj.load_state_dict(sd, strict=True)
audio_proj.eval()
print(f"Loaded AudioProjection: {AUDIO_PROJ_PATH}")

ck = torch.load(VIDEO_PROJ_PATH, map_location=DEVICE)
sd = ck.get("model_state_dict", ck)
video_proj.load_state_dict(sd, strict=False)
video_proj.eval()
print(f"Loaded VideoFrameProjection: {VIDEO_PROJ_PATH}")

print(f"AudioProjection trainable params: {sum(p.numel() for p in audio_proj.parameters()):,}")
print(f"VideoFrameProjection trainable params: {sum(p.numel() for p in video_proj.parameters() if p.requires_grad):,}")



## Challenge 1 — Multimodal representation (CREMA-D audio ↔ video)

Symmetric InfoNCE between **projected** audio and video embeddings; P@K/MRR on paired clips (same stem).


In [ ]:
def symmetric_infonce_loss(e_A: torch.Tensor, e_B: torch.Tensor, tau: float = TAU) -> torch.Tensor:
    e_A = F.normalize(e_A, p=2, dim=-1)
    e_B = F.normalize(e_B, p=2, dim=-1)
    B = e_A.shape[0]
    S = (e_A @ e_B.T) / tau
    labels = torch.arange(B, device=e_A.device)
    return 0.5 * (F.cross_entropy(S, labels) + F.cross_entropy(S.T, labels))


def precision_at_k_mrr(
    query_embs: torch.Tensor,
    corpus_embs: torch.Tensor,
    relevant_ids: List[List[int]],
    k_vals: List[int] = (1, 3, 5),
) -> Dict:
    scores = query_embs @ corpus_embs.T
    ranked = scores.argsort(dim=1, descending=True)
    hits = {k: 0 for k in k_vals}
    rr = 0.0
    for q, rel in enumerate(relevant_ids):
        rel_set = set(rel)
        top = ranked[q].tolist()
        for k in k_vals:
            if any(i in rel_set for i in top[:k]):
                hits[k] += 1
        for rank, i in enumerate(top, 1):
            if i in rel_set:
                rr += 1 / rank
                break
    n = max(len(relevant_ids), 1)
    out = {f"P@{k}": hits[k] / n for k in k_vals}
    out["MRR"] = rr / n
    return out


# ── CREMA cache (real clips only; loaded in setup) ─────────────
assert crema_cache is not None and len(crema_cache) >= 32
keys = list(crema_cache.keys())
# Prefer clips with valid video frames (Pipeline D convention)
keys_vid = [k for k in keys if crema_cache[k].get("has_video", True)]
if len(keys_vid) < 32:
    raise RuntimeError("Need at least 32 clips with has_video=True in crema_cache for evaluation.")

random.seed(SEED)
random.shuffle(keys_vid)
N_EVAL = min(256, len(keys_vid))
subset = keys_vid[:N_EVAL]

audio_raw = torch.stack([crema_cache[k]["audio_emb"] for k in subset]).to(DEVICE)
frames = torch.stack([crema_cache[k]["frame_tensor"] for k in subset]).to(DEVICE)

audio_proj.eval()
video_proj.eval()
with torch.no_grad():
    e_a = audio_proj(audio_raw)
    e_v = video_proj(frames)

B32 = min(32, e_a.shape[0])
infonce_val = symmetric_infonce_loss(e_a[:B32], e_v[:B32]).item()
relevant = [[i] for i in range(N_EVAL)]
metrics_c1 = precision_at_k_mrr(e_a.cpu(), e_v.cpu(), relevant)

print("\n[C1] Representation — Audio → Video retrieval (CREMA-D)")
print(f"  InfoNCE (batch {B32}): {infonce_val:.4f}  (random baseline ~log(B)={math.log(B32):.2f})")
for k, v in metrics_c1.items():
    print(f"  {k:5s}: {v * 100:.1f}%")

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
S = (e_a[:16] @ e_v[:16].T).detach().cpu().numpy()
im = ax.imshow(S, cmap="Blues", vmin=-0.5, vmax=1.0)
ax.set_title("[C1] Audio–video similarity (first 16 clips)")
ax.set_xlabel("Video index")
ax.set_ylabel("Audio index")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()



## Challenge 2 — “Translation” as label fidelity (emotion words)

CMU-MOSI transcript WER/BLEU is replaced by **word-level error** on **emotion names** (Pipeline D / Phase 2–3 use the same 6 labels). This keeps the same WER/BLEU machinery while matching CREMA-D.


In [ ]:
def compute_wer(reference: str, hypothesis: str) -> float:
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    N = len(ref)
    if N == 0:
        return 0.0
    dp = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1):
        dp[i][0] = i
    for j in range(len(hyp) + 1):
        dp[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            dp[i][j] = dp[i - 1][j - 1] if ref[i - 1] == hyp[j - 1] else 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[len(ref)][len(hyp)] / N


def compute_bleu(reference: str, hypothesis: str, n: int = 2) -> float:
    ref, hyp = reference.lower().split(), hypothesis.lower().split()
    if not hyp:
        return 0.0
    log_sum = 0.0
    for k in range(1, n + 1):
        ref_ng = Counter(tuple(ref[i : i + k]) for i in range(len(ref) - k + 1))
        hyp_ng = Counter(tuple(hyp[i : i + k]) for i in range(len(hyp) - k + 1))
        matches = sum(min(hyp_ng[ng], ref_ng.get(ng, 0)) for ng in hyp_ng)
        denom = max(1, len(hyp) - k + 1)
        log_sum += math.log(max(matches / denom, 1e-10))
    bp = min(1.0, math.exp(1 - len(ref) / max(len(hyp), 1)))
    return bp * math.exp(log_sum / n)


# Real emotion labels from crema_cache only (no simulated ASR)
_emo_keys = [k for k in crema_cache.keys() if "emotion" in crema_cache[k]]
random.seed(42)
random.shuffle(_emo_keys)
pairs = min(200, len(_emo_keys) - 1)
refs = [crema_cache[_emo_keys[i]]["emotion"] for i in range(pairs)]
hyps = [crema_cache[_emo_keys[i + 1]]["emotion"] for i in range(pairs)]
wers = [compute_wer(r, h) for r, h in zip(refs, hyps)]
bleus = [compute_bleu(r, h) for r, h in zip(refs, hyps)]

print("\n[C2] WER / BLEU on consecutive real CREMA emotion labels (paired clips)")
print(f"  Pairs: {len(refs)}")
print(f"  Mean WER: {float(np.mean(wers)) * 100:.1f}%")
print(f"  Mean BLEU-2: {float(np.mean(bleus)):.4f}")
for i in range(min(3, len(refs))):
    print(f'  REF: "{refs[i]}"  vs  HYP: "{hyps[i]}"  WER={wers[i]*100:.1f}%')


## Challenge 3 — Multimodal fusion (audio + video → emotion)

Pipeline E’s 3-way sentiment fusion is not applicable without text embeddings. Here we train a **small head** on **frozen** `audio_proj` / `video_proj` outputs (concat → MLP → 6 logits), mirroring the fusion idea on CREMA-D.


In [ ]:
class AudioVideoFusionHead(nn.Module):
    def __init__(self, embed_dim: int = EMBED_DIM, n_classes: int = 6):
        super().__init__()
        d = embed_dim * 2
        self.net = nn.Sequential(
            nn.Linear(d, embed_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, n_classes),
        )

    def forward(self, a: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        x = torch.cat([a, v], dim=-1)
        return self.net(x)


class CremaTensorDataset(Dataset):
    def __init__(self, keys: List[str], cache: dict):
        self.keys = keys
        self.cache = cache

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx: int):
        k = self.keys[idx]
        v = self.cache[k]
        return v["audio_emb"], v["frame_tensor"], int(v["label"])


def collate_crema(batch):
    a = torch.stack([b[0] for b in batch])
    f = torch.stack([b[1] for b in batch])
    y = torch.tensor([b[2] for b in batch], dtype=torch.long)
    return a, f, y


# ── Train / test split by actor (approx. speaker-independent) ─
actor_to_keys = defaultdict(list)
for k in keys_vid:
    actor_to_keys[int(crema_cache[k]["actor_id"])].append(k)
actors = sorted(actor_to_keys.keys())
random.seed(SEED)
random.shuffle(actors)
n_test = max(1, len(actors) // 5)
test_actors = set(actors[:n_test])
train_keys = [k for k in keys_vid if int(crema_cache[k]["actor_id"]) not in test_actors]
test_keys = [k for k in keys_vid if int(crema_cache[k]["actor_id"]) in test_actors]
if len(train_keys) < 64 or len(test_keys) < 8:
    split = int(0.85 * len(keys_vid))
    train_keys = keys_vid[:split]
    test_keys = keys_vid[split:]

train_ds = CremaTensorDataset(train_keys, crema_cache)
test_ds = CremaTensorDataset(test_keys, crema_cache)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_crema)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_crema)

fusion_head = AudioVideoFusionHead().to(DEVICE)
opt = torch.optim.AdamW(fusion_head.parameters(), lr=1e-3, weight_decay=0.01)

audio_proj.eval()
video_proj.eval()
for p in audio_proj.parameters():
    p.requires_grad = False
for p in video_proj.parameters():
    p.requires_grad = False

EPOCHS = 3
for epoch in range(EPOCHS):
    fusion_head.train()
    losses = []
    for a_b, f_b, y_b in train_loader:
        a_b, f_b, y_b = a_b.to(DEVICE), f_b.to(DEVICE), y_b.to(DEVICE)
        with torch.no_grad():
            ea = audio_proj(a_b)
            ev = video_proj(f_b)
        logits = fusion_head(ea, ev)
        loss = F.cross_entropy(logits, y_b)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    print(f"  [C3] epoch {epoch+1}/{EPOCHS}  loss={float(np.mean(losses)):.4f}")


@torch.no_grad()
def eval_fusion():
    fusion_head.eval()
    correct = total = 0
    for a_b, f_b, y_b in test_loader:
        a_b, f_b, y_b = a_b.to(DEVICE), f_b.to(DEVICE), y_b.to(DEVICE)
        ea = audio_proj(a_b)
        ev = video_proj(f_b)
        pred = fusion_head(ea, ev).argmax(dim=-1)
        correct += (pred == y_b).sum().item()
        total += y_b.numel()
    return correct / max(total, 1)


acc_fusion = eval_fusion()
print(f"\n[C3] Fusion — held-out accuracy (6-way emotion): {acc_fusion * 100:.1f}%")
print(f"  Train clips: {len(train_keys):,} | Test clips: {len(test_keys):,}")


## Challenge 4 — Co-learning (audio ↔ video on the same sphere)

Positive pairs: same clip; negative pairs: shuffled video. Same idea as CMU-MOSI text ↔ audio, with CREMA modalities.


In [ ]:
N_DEMO = min(200, len(subset))
idxs = subset[:N_DEMO]
audio_raw_d = torch.stack([crema_cache[k]["audio_emb"] for k in idxs]).to(DEVICE)
frames_d = torch.stack([crema_cache[k]["frame_tensor"] for k in idxs]).to(DEVICE)
audio_proj.eval()
video_proj.eval()
with torch.no_grad():
    t_a = audio_proj(audio_raw_d)
    t_v = video_proj(frames_d)

sim_pos = (t_a * t_v).sum(dim=-1).cpu().numpy()
perm = torch.randperm(N_DEMO)
sim_neg = (t_a * t_v[perm]).sum(dim=-1).cpu().numpy()

print("[C4] Co-learning — CREMA-D audio ↔ video")
print(f"  Same-clip cosine   : {sim_pos.mean():.4f} ± {sim_pos.std():.4f}")
print(f"  Shuffled cosine    : {sim_neg.mean():.4f} ± {sim_neg.std():.4f}")
print(f"  Gap (pos - neg)    : {sim_pos.mean() - sim_neg.mean():+.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sim_pos, bins=30, alpha=0.7, color="#27ae60", label="Same clip")
ax.hist(sim_neg, bins=30, alpha=0.7, color="#e74c3c", label="Shuffled video")
ax.set_title("[C4] Audio–video cosine similarity (projected)")
ax.legend()
plt.tight_layout()
plt.show()


## Challenge 5 — Alignment (segment-level)

Each cache entry is one CREMA-D clip; audio and video are **inherently aligned** in time (middle frame vs full-clip audio embedding). We compare aligned vs mismatched pairs like the CMU-MOSI notebook.


In [ ]:
N_ALIGN = min(64, len(subset))
audio_raw_al = torch.stack([crema_cache[k]["audio_emb"] for k in subset[:N_ALIGN]]).to(DEVICE)
frames_al = torch.stack([crema_cache[k]["frame_tensor"] for k in subset[:N_ALIGN]]).to(DEVICE)
with torch.no_grad():
    e_at = audio_proj(audio_raw_al)
    e_vt = video_proj(frames_al)

sim_aligned = (e_at * e_vt).sum(-1).cpu().numpy()
perm2 = torch.randperm(N_ALIGN)
while perm2.numel() > 1 and (perm2 == torch.arange(N_ALIGN)).all():
    perm2 = torch.randperm(N_ALIGN)
sim_mis = (e_at * e_vt[perm2]).sum(-1).cpu().numpy()

print("[C5] Alignment — CREMA-D")
print(f"  Aligned pairs   : {sim_aligned.mean():.4f} ± {sim_aligned.std():.4f}")
print(f"  Mismatched pairs: {sim_mis.mean():.4f} ± {sim_mis.std():.4f}")


def soft_alignment(H_A: torch.Tensor, H_B: torch.Tensor, d_k: int = 64) -> Tuple[torch.Tensor, torch.Tensor]:
    """Scaled dot-product attention on real projected embeddings (no random weight matrices)."""
    d = H_A.shape[-1]
    scale = math.sqrt(float(d))
    logits = (H_A @ H_B.T) / scale
    alpha = torch.softmax(logits, dim=1)
    context = alpha @ H_B
    return alpha, context


def dtw_alignment(H_A: torch.Tensor, H_B: torch.Tensor) -> Tuple[np.ndarray, float]:
    T_A, T_B = H_A.shape[0], H_B.shape[0]
    D = (1.0 - (F.normalize(H_A, dim=-1) @ F.normalize(H_B, dim=-1).T)).cpu().numpy()
    INF = float("inf")
    C = np.full((T_A + 1, T_B + 1), INF)
    C[0, 0] = 0.0
    for i in range(1, T_A + 1):
        for j in range(1, T_B + 1):
            C[i, j] = D[i - 1, j - 1] + min(C[i - 1, j], C[i, j - 1], C[i - 1, j - 1])
    path = []
    i, j = T_A, T_B
    while i > 0 and j > 0:
        path.append((i - 1, j - 1))
        moves = [(C[i - 1, j], (i - 1, j)), (C[i, j - 1], (i, j - 1)), (C[i - 1, j - 1], (i - 1, j - 1))]
        _, (i, j) = min(moves, key=lambda x: x[0])
    return np.array(list(reversed(path))), float(C[T_A, T_B])


H_A_demo = e_at[:8]
H_B_demo = e_vt[:8]
alpha, _ = soft_alignment(H_A_demo, H_B_demo)
dtw_path, dtw_cost = dtw_alignment(e_at[:6].cpu(), e_vt[:8].cpu())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(sim_aligned, bins=20, alpha=0.7, color="#27ae60", label="Aligned")
axes[0].hist(sim_mis, bins=20, alpha=0.7, color="#e74c3c", label="Misaligned")
axes[0].legend()
axes[0].set_title("[C5] Pairwise cosine")
im = axes[1].imshow(alpha.detach().cpu().numpy(), cmap="Blues", aspect="auto")
axes[1].set_title("[C5] Soft alignment (toy)")
plt.colorbar(im, ax=axes[1], fraction=0.046)
cost_mat = (1.0 - (F.normalize(e_at[:6], dim=-1) @ F.normalize(e_vt[:8], dim=-1).T)).cpu().numpy()
axes[2].imshow(cost_mat, cmap="hot_r", aspect="auto", origin="lower")
if len(dtw_path) > 0:
    axes[2].plot(dtw_path[:, 1], dtw_path[:, 0], "b-o", markersize=4, label=f"DTW cost={dtw_cost:.2f}")
axes[2].set_title("[C5] DTW (toy)")
axes[2].legend()
plt.tight_layout()
plt.show()



## Unified demo — emotion prediction via fusion head

Runs a short table on held-out clips: true emotion vs argmax prediction.


In [ ]:
@torch.no_grad()
def run_unified_demo(n: int = 10):
    fusion_head.eval()
    audio_proj.eval()
    video_proj.eval()
    print(f"{'Clip':<22} {'True':<12} {'Pred':<12} {'OK'}")
    print("-" * 52)
    ok = 0
    for k in test_keys[:n]:
        v = crema_cache[k]
        a = v["audio_emb"].unsqueeze(0).to(DEVICE)
        f = v["frame_tensor"].unsqueeze(0).to(DEVICE)
        ea = audio_proj(a)
        ev = video_proj(f)
        pred = fusion_head(ea, ev).argmax(dim=-1).item()
        true = int(v["label"])
        good = pred == true
        ok += int(good)
        short = k[-20:] if len(k) > 20 else k
        print(
            f"{short:<22} {CREMA_LABEL_TO_EMOTION[true]:<12} {CREMA_LABEL_TO_EMOTION[pred]:<12} {'Y' if good else 'N'}"
        )
    print(f"\nMini-batch accuracy: {ok}/{min(n, len(test_keys))}")


print("=" * 52)
print("UNIFIED — CREMA-D (Pipeline D projections + fusion head)")
print("=" * 52)
run_unified_demo(min(10, len(test_keys)))

# Misalignment stress test: shuffle video inputs
preds_ok, preds_bad = [], []
with torch.no_grad():
    for i, k in enumerate(test_keys[: min(64, len(test_keys))]):
        v = crema_cache[k]
        a = v["audio_emb"].unsqueeze(0).to(DEVICE)
        f = v["frame_tensor"].unsqueeze(0).to(DEVICE)
        f_bad = crema_cache[test_keys[(i + 1) % len(test_keys)]]["frame_tensor"].unsqueeze(0).to(DEVICE)
        ea = audio_proj(a)
        preds_ok.append(fusion_head(ea, video_proj(f)).argmax(dim=-1).item())
        preds_bad.append(fusion_head(ea, video_proj(f_bad)).argmax(dim=-1).item())
true_labs = [int(crema_cache[k]["label"]) for k in test_keys[: len(preds_ok)]]
acc_a = np.mean(np.array(preds_ok) == np.array(true_labs))
acc_b = np.mean(np.array(preds_bad) == np.array(true_labs))
print(f"\nAligned video   acc: {acc_a * 100:.1f}%")
print(f"Misaligned video acc: {acc_b * 100:.1f}%")


## Summary

| Challenge | CREMA-D / Pipeline D variant |
|---|---|
| C1 | Audio ↔ video InfoNCE + P@K on paired clips |
| C2 | WER/BLEU on emotion words (label fidelity proxy) |
| C3 | Frozen projections + small concat fusion head (6-way) |
| C4 | Same-clip vs shuffled video cosine gap |
| C5 | Aligned vs mismatched pairs + toy soft attention / DTW |

**Run in Colab** with Drive mounted so `crema_cache.pt` and `audio_projection_cremad.pth` load from `models/miva_knight_pipelineD/`.
